In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
import pandas as pd

from os.path import join
import os
from functools import partial
import pathlib
from pathlib import Path
import shutil

from joblib import Parallel, delayed
import joblib

import gc
import subprocess

from credit.pbs import get_num_cpus

In [ ]:
#parameters
forecast_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/wx_inverted/forecasts/2022-07-01T00:25:06"
channels = [7, 8, 9, 10, 13]
num_cpus = 2
test = 0
config = None
delay = 50
dpi = 300
max_steps = 12

In [ ]:
if c13_only:
    channels = [13]
else:
    channels = [7, 8, 9, 10, 13]

In [ ]:
def plot_forecast_and_diff(ds, true_ds, init_ds, forecast_dir, channels, img_idx=-1, diff_max=25):
    true_ds["t"] = ds.t
    
    for channel in channels:
        fig = plt.figure(figsize=(12, 5))
        gs = gridspec.GridSpec(
            1, 5,
            width_ratios=[1, 1, 0.1, 1, 0.1],
            wspace=0.1,
            figure=fig,
        )

        i=0
        # Create axes
        ax1 = fig.add_subplot(gs[i, 0], projection=ccrs.PlateCarree())
        ax2 = fig.add_subplot(gs[i, 1], projection=ccrs.PlateCarree())
        cax = fig.add_subplot(gs[i, 2])  # colorbar axis (between 2nd & 3rd)
        ax3 = fig.add_subplot(gs[i, 3], projection=ccrs.PlateCarree())
        cax2 = fig.add_subplot(gs[i, 4])

        # Load the raw prediction
        pred = ds.sel(channel=channel).BT_or_R #.isel(latitude=slice(11,-10), longitude=slice(19,-18))
        
        # --- THE FIX FOR ENSEMBLES ---
        if "ensemble_member_label" in pred.dims:
            # Option A: Plot the ensemble mean (Recommended)
            pred = pred.mean(dim="ensemble_member_label")
            
            # Option B: If you prefer to plot just one specific member:
            # pred = pred.isel(ensemble_member_label=3)
        # -----------------------------
        
        # Now squeeze to remove time/channel dims, making it strictly 2D
        pred = pred.squeeze()
        true = true_ds.sel(channel=channel).BT_or_R.squeeze()
        init = init_ds.sel(channel=channel).BT_or_R.squeeze()
        
        if channel > 4:
            pred.plot(ax=ax1, transform=ccrs.PlateCarree(), cmap="Spectral_r", vmax=300, vmin=200,  add_colorbar=False)
            im2 = true.plot(ax=ax2, transform=ccrs.PlateCarree(), cmap="Spectral_r", vmax=300, vmin=200, add_colorbar=False)
            
            diff = pred - true
            diffbar = diff.plot(ax=ax3, transform=ccrs.PlateCarree(), cmap="seismic", vmax=diff_max, vmin=-diff_max, add_colorbar=False)
            ax3.set_title("Prediction - Observations")
        else: #channel 4
            vmin, vmax = 0.0, 0.2
            pred.plot(ax=ax1, transform=ccrs.PlateCarree(), cmap="Spectral_r", vmax=vmax, vmin=vmin, add_colorbar=False)
            im2 = true.plot(ax=ax2, transform=ccrs.PlateCarree(), cmap="Spectral_r", vmax=vmax, vmin=vmin, add_colorbar=False)
    
            diff = pred - true
            diffbar = diff.plot(ax=ax3, transform=ccrs.PlateCarree(), cmap="seismic", vmax=0.1, vmin=-0.1, add_colorbar=False)
        
        ax1.set_title("Prediction" if img_idx > 0 else "IC")
        ax2.set_title("Observations")
        ax3.set_title("Prediction - Observations")

        ax1.set_yticks(list(range(-50,51,25)))

        for ax in [ax1, ax2, ax3]:
            ax.add_feature(cfeature.COASTLINE)
            ax.set_xticks(list(range(-120,-29,30)))
    
        for ax in [cax, cax2]:
            ax.axis('off')
            ax.get_xaxis().set_ticks([])
            ax.get_yaxis().set_ticks([])
            
        cbar = fig.colorbar(im2, ax=cax, orientation='vertical', fraction = 1, shrink=0.75)
        cbar = fig.colorbar(diffbar, ax=cax2, orientation='vertical', fraction = 1, shrink=0.75)
    
        # fig.suptitle(f"Channel {channel}, forecast step {img_idx:02}\n{pd.Timestamp(pred.t.values[0]).strftime('%Y-%m-%dT%H:%M:%S')}")
        fig.suptitle(f"Channel {channel}, forecast step {img_idx:02}\n{pd.Timestamp(pred.t.values.item()).strftime('%Y-%m-%dT%H:%M:%S')}")

        figname = f"C{channel:02}_step{img_idx:02}.png"

        path1 = pathlib.Path(forecast_dir)
        time_str = path1.parent.name if not path1.is_dir() else path1.name

        forecast_parent_dir = Path(forecast_dir).parent
        save_dir = join(forecast_parent_dir, "gifs", f"gifs_{time_str}/C{channel:02}")
        
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(join(save_dir, figname), format="png", dpi=dpi)

        plt.close(fig)

In [ ]:
num_cpus = max(1, get_num_cpus() - 1)

print(f"using {num_cpus} cpus")
print(f"processing {forecast_dir}")

files = sorted([f for f in os.listdir(forecast_dir) if os.path.isfile(join(forecast_dir, f))])[:max_steps]

if test:
    print("WARNING: test mode!")
    files = files[:2]
    num_cpus = 2
    
def make_gif(forecast_dir, channels, file_and_index):
    k, file = file_and_index
    
    img_idx = k + 1
    
    zarr_ds = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr", consolidated=False).drop_duplicates(
        dim="t"
    ).sortby(
        "t"
    ).transpose(
        "t", "channel", "latitude", "longitude")
    
    # get init data
    path1 = pathlib.Path(forecast_dir)
    time_str = path1.parent.name if not path1.is_dir() else path1.name
    init_time = pd.Timestamp(time_str)
    init_ds = zarr_ds.sel(t=[init_time], method="nearest")

    if img_idx == 0: # plot the init image
        plot_forecast_and_diff(init_ds, init_ds, init_ds, forecast_dir, channels, img_idx)
        gc.collect()
        return

    # plot the forecast and obs
    file = join(forecast_dir, file)
    ds = xr.open_dataset(file) # get prediction
    true_ds = zarr_ds.sel(t=ds.t, method="nearest")

    plot_forecast_and_diff(ds, true_ds, init_ds, forecast_dir, channels, img_idx)
    
    gc.collect()

f = partial(make_gif, forecast_dir, channels)

enum_files = [(-1, None)] + list(enumerate(files))
print(files)

result = Parallel(n_jobs = num_cpus)(delayed(f)(file_and_index)
                            for file_and_index in enum_files)

using 1 cpus
processing /glade/derecho/scratch/dkimpara/goes_10km_train/wx_inverted/forecasts/2022-07-01T00:25:06


: 

In [ ]:
path1 = pathlib.Path(forecast_dir)
time_str = path1.name
gif_dir = join(path1.parent, "gifs", f"gifs_{time_str}")


for channel in channels:
    res = subprocess.run(
                f"magick -delay {delay} -loop 0 {join(gif_dir, f'C{channel:02}/*.png')} {join(gif_dir, f'C{channel:02}.gif')}",
                shell=True,
                capture_output=True,
                encoding="utf-8",
            )
    print(f"created gif for C{channel:02}")
    shutil.rmtree(join(gif_dir, f'C{channel:02}'))
